# Balance des Paiements CEMAC  Source BEAC locale (2020-2023)

Meme methodologie et memes 10 metriques que `bop_fmi.ipynb` (source FMI), pour les
pays de la CEMAC disposant de releves BEAC locaux dans `data/2020-2023-CEMAC/` : Congo,
Gabon, Guinee equatoriale, Tchad, Republique centrafricaine, Cameroun. PIB par secteur :
`data/pib_cemac_par_secteur/`.

**Cameroun couvert par les deux pipelines** : ce notebook le traite aussi (source BEAC
locale), en plus de `bop_fmi.ipynb` (source FMI) -- volontaire, pour permettre une
comparaison BEAC vs FMI sur ce pays. Les CSV de sortie des deux pipelines restent
distincts (suffixe `_BEAC` ici), donc pas d'ecrasement mutuel.

**Devise** : montants sources en Millions/FCFA (BDP) et Milliards/FCFA (PIB), convertis en
millions USD via un taux fixe fourni (1 XAF = 0,001771 USD) -- pas de taux annuel variable
disponible, donc pas de conversion par annee.

---
## 1. Configuration

In [1]:
import os
import unicodedata
from pathlib import Path
import pandas as pd
import openpyxl
import plotly.graph_objects as go


In [2]:
BASE = Path(os.path.abspath('')).parent
DIR_BRUT_CEMAC = BASE / 'data' / '2020-2023-CEMAC'
DIR_PIB_CEMAC = BASE / 'data' / 'pib_cemac_par_secteur'
DIR_SORTIE = BASE / 'data'
FIGURES_DIR = BASE / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Taux de change fixe fourni par l'utilisateur (pas de conversion inventee) : 1 XAF = 0,001771 USD.
# Applique uniformement sur 2020-2023 (pas de taux annuel distinct disponible).
TAUX_XAF_USD = 0.001771

ANNEES_ETUDE = [2020, 2021, 2022, 2023]

# Un seul fichier source par (pays, annee) -- resolu manuellement apres verification :
# - Les variantes "(1)" a cote d'un fichier de meme nom sont des doublons strictement
#   identiques (meme taille en octets, verifie) : on garde arbitrairement une seule copie.
# - RCA (Centrafrique) : les fichiers BDP_RCA_2020/2021/2022_16092025.xlsx sont corrompus
#   (erreur zip, illisibles). 2020 et 2021 sont remplaces par des fichiers alternatifs
#   (memes chiffres a des arrondis pres) ; aucun fichier lisible n'existe pour 2022 --
#   laisse absent plutot que d'inventer une valeur.
FICHIERS_BOP_CEMAC = {
    'COG': {
        2020: 'BDP_CONGO_2020_15092025.xlsx',
        2021: 'BDP_CONGO_2021_15092025.xlsx',
        2022: 'BDP_CONGO_2022_15092025-1 (1).xlsx',
        2023: 'BDP_CONGO_2023_15092025.xlsx',
    },
    'GAB': {
        2020: 'BDP_GABON_2020_11102025.xlsx',
        2021: 'BDP_GABON_2021_11102025 (1).xlsx',
        2022: 'BDP_GABON_2022_11102025.xlsx',
        2023: 'BDP_GABON_2023_11102025.xlsx',
    },
    'GNQ': {
        2020: 'BDP_GUINEE_EQUATORIALE_2020_08122025.xlsx',
        2021: 'BDP_GUINEE_EQUATORIALE_2021_08122025.xlsx',
        2022: 'BDP_GUINEE_EQUATORIALE_2022_08122025.xlsx',
        2023: 'BDP_GUINEE_EQUATORIALE_2023_08122025.xlsx',
    },
    'TCD': {
        2020: 'BDP_TCHAD_2020.xlsx',
        2021: 'BDP_TCHAD_2021 (1).xlsx',
        2022: 'BDP_TCHAD_2022.xlsx',
        2023: 'BDP_TCHAD_2023.xlsx',
    },
    'CAF': {
        2020: 'Balance-des-Paiements-Analytique-2020-RCA.xlsx',
        2021: 'Balance-des-Paiements-Analytique-2021-RCA.xlsx',
        # 2022 : aucun fichier lisible disponible -- absent intentionnellement.
        2023: 'BDP_RCA_2023_16092025-1.xlsx',
    },
    'CMR': {
        2020: 'Balance-des-Paiments-Analytique-2020-Cameroun.xlsx',
        2021: 'Balance-des-Paiments-Analytique-2021-Cameroun.xlsx',
        2022: 'Balance-des-Paiments-Analytique-2022-Cameroun.xlsx',
        # 2023: 'Balance-des-Paiments-Analytique-2023-Cameroun.xlsx',
    }
}

PAYS_CEMAC_LOCAL = {
    'COG': 'Congo',
    'GAB': 'Gabon',
    'GNQ': 'Guinée équatoriale',
    'TCD': 'Tchad',
    'CAF': 'République centrafricaine',
    'CMR': 'Cameroun',
}

# PIB par secteur (deor_<pays>.xlsx), une valeur "PIB aux prix courants du marche" par pays.
FICHIERS_PIB_CEMAC = {
    'COG': 'deor_congo-1.xlsx',
    'GAB': 'deor_gabon-1.xlsx',
    'GNQ': 'deor_guinee-equatoriale-1.xlsx',
    'TCD': 'deor_tchad-1.xlsx',
    'CAF': 'deor_rca-1.xlsx',
    'CMR': 'deor_cameroun-1.xlsx',
}

print(f"{len(FICHIERS_BOP_CEMAC)} pays CEMAC (source BEAC locale), annees d'etude : {ANNEES_ETUDE}")
for code, nom in PAYS_CEMAC_LOCAL.items():
    annees_dispo = sorted(FICHIERS_BOP_CEMAC[code])
    print(f"  {nom:<28} ({code}) : {annees_dispo}")


6 pays CEMAC (source BEAC locale), annees d'etude : [2020, 2021, 2022, 2023]
  Congo                        (COG) : [2020, 2021, 2022, 2023]
  Gabon                        (GAB) : [2020, 2021, 2022, 2023]
  Guinée équatoriale           (GNQ) : [2020, 2021, 2022, 2023]
  Tchad                        (TCD) : [2020, 2021, 2022, 2023]
  République centrafricaine    (CAF) : [2020, 2021, 2023]
  Cameroun                     (CMR) : [2020, 2021, 2022]


---
## 2. Lecture des fichiers BDP BEAC (feuille ANALYTIQUE)

Les fichiers `BDP_<PAYS>_<ANNEE>*.xlsx` presentent une feuille `ANALYTIQUE` avec un
libelle hierarchique (indentation) et 3 colonnes Credit/Debit/Solde. Certains fichiers
alternatifs (RCA 2020/2021) ont une mise en page differente (colonnes decalees, pas
d'indentation) -- la detection d'en-tete est generique (recherche par contenu de
cellule, pas par position fixe) pour geradir les deux cas uniformement.

In [3]:
def _normaliser(texte):
    """Minuscule, espace insecable -> espace normal, espaces multiples ecrases."""
    if texte is None:
        return ''
    return str(texte).replace('\xa0', ' ').strip()


def detecter_entete(ws):
    """
    Cherche la ligne d'en-tete contenant Libelle/Credit/Debit/Solde -- la position des
    colonnes varie selon le fichier source (feuille ANALYTIQUE standard : colonnes
    B/F/G/H ; fichiers alternatifs RCA 2020/2021 : colonnes A/B/C/D ou A/.../J/K/L).
    Detection generique par contenu de cellule plutot que position fixe.
    """
    for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
        col_libelle = col_credit = col_debit = col_solde = None
        for j, val in enumerate(row):
            v = _normaliser(val).lower()
            if not v:
                continue
            if v.startswith('libell'):
                col_libelle = j
            elif v.startswith('crédit') or v.startswith('credit'):
                col_credit = j
            elif v.startswith('débit') or v.startswith('debit'):
                col_debit = j
            elif v.startswith('solde'):
                col_solde = j
        if None not in (col_libelle, col_credit, col_debit, col_solde):
            return i, col_libelle, col_credit, col_debit, col_solde
    raise ValueError("En-tete Libelle/Credit/Debit/Solde introuvable dans cette feuille.")


def choisir_feuille(wb):
    """La feuille 'ANALYTIQUE' quand elle existe (fichiers BEAC standard) ; sinon
    l'unique feuille du classeur (fichiers alternatifs RCA a feuille unique)."""
    if 'ANALYTIQUE' in wb.sheetnames:
        return wb['ANALYTIQUE']
    if len(wb.sheetnames) == 1:
        return wb[wb.sheetnames[0]]
    raise ValueError(f"Feuille ANALYTIQUE absente et plusieurs feuilles ambigues : {wb.sheetnames}")


def lire_bdp_analytique(chemin):
    """
    Lit un fichier BDP BEAC (feuille ANALYTIQUE ou equivalent) et retourne un DataFrame
    long : LIBELLE (nettoye), CREDIT, DEBIT, SOLDE (en millions de FCFA, tels quels dans
    le fichier source), ligne par ligne dans l'ordre du fichier (l'ordre porte la
    hierarchie : cf. extraire_agregats qui s'appuie sur les en-tetes de section pour
    circonscrire ses recherches, plutot que sur la profondeur d'indentation -- certains
    fichiers alternatifs n'ont plus d'indentation du tout apres nettoyage en amont).
    """
    wb = openpyxl.load_workbook(chemin, data_only=True)
    ws = choisir_feuille(wb)
    ligne_entete, c_lib, c_cr, c_db, c_sd = detecter_entete(ws)

    lignes = []
    for row in ws.iter_rows(min_row=ligne_entete + 1, values_only=True):
        libelle = _normaliser(row[c_lib]) if c_lib < len(row) else ''
        if not libelle:
            continue
        credit = row[c_cr] if c_cr < len(row) else None
        debit = row[c_db] if c_db < len(row) else None
        solde = row[c_sd] if c_sd < len(row) else None
        lignes.append({
            'LIBELLE': libelle,
            'CREDIT': pd.to_numeric(credit, errors='coerce'),
            'DEBIT': pd.to_numeric(debit, errors='coerce'),
            'SOLDE': pd.to_numeric(solde, errors='coerce'),
        })
    return pd.DataFrame(lignes)


---
## 3. Consolidation brute (tous pays x toutes annees disponibles)

In [4]:
lignes_brut = []
echecs_lecture = []

for code, fichiers_annee in FICHIERS_BOP_CEMAC.items():
    for annee, nom_fichier in fichiers_annee.items():
        chemin = DIR_BRUT_CEMAC / nom_fichier
        try:
            df_pays_annee = lire_bdp_analytique(chemin)
        except Exception as e:
            echecs_lecture.append((code, annee, nom_fichier, str(e)))
            print(f"ECHEC lecture {code} {annee} ({nom_fichier}) : {e}")
            continue
        df_pays_annee.insert(0, 'ANNEE', annee)
        df_pays_annee.insert(0, 'COUNTRY_CODE', code)
        lignes_brut.append(df_pays_annee)

df_brut_cemac = pd.concat(lignes_brut, ignore_index=True)
df_brut_cemac['CREDIT_USD_M'] = df_brut_cemac['CREDIT'] * TAUX_XAF_USD
df_brut_cemac['DEBIT_USD_M'] = df_brut_cemac['DEBIT'] * TAUX_XAF_USD
df_brut_cemac['SOLDE_USD_M'] = df_brut_cemac['SOLDE'] * TAUX_XAF_USD
df_brut_cemac = df_brut_cemac.rename(columns={'CREDIT': 'CREDIT_FCFA_M', 'DEBIT': 'DEBIT_FCFA_M', 'SOLDE': 'SOLDE_FCFA_M'})

chemin_brut_csv = DIR_SORTIE / 'BOP_CEMAC_BEAC_BRUT.csv'
df_brut_cemac.to_csv(chemin_brut_csv, index=False)

print(f"{chemin_brut_csv} : {len(df_brut_cemac)} lignes ({df_brut_cemac['COUNTRY_CODE'].nunique()} pays x annees).")
if echecs_lecture:
    print(f"\n{len(echecs_lecture)} fichier(s) non lus :")
    for code, annee, nom_fichier, err in echecs_lecture:
        print(f"  {code} {annee} ({nom_fichier}) : {err}")
couverture = df_brut_cemac.groupby('COUNTRY_CODE')['ANNEE'].apply(lambda s: sorted(s.unique()))
print("\nCouverture reelle par pays :")
for code, annees in couverture.items():
    print(f"  {PAYS_CEMAC_LOCAL[code]:<28} ({code}) : {annees}")


m:\Travail\BEAC\balance_de_paiement\data\BOP_CEMAC_BEAC_BRUT.csv : 10433 lignes (6 pays x annees).

Couverture reelle par pays :
  République centrafricaine    (CAF) : [2020, 2021, 2023]
  Cameroun                     (CMR) : [2020, 2021, 2022]
  Congo                        (COG) : [2020, 2021, 2022, 2023]
  Gabon                        (GAB) : [2020, 2021, 2022, 2023]
  Guinée équatoriale           (GNQ) : [2020, 2021, 2022, 2023]
  Tchad                        (TCD) : [2020, 2021, 2022, 2023]


---
## 4. Extraction des agregats BPM6 (equivalent de BOP_PAR_COMPTE.csv, section 14 du
notebook principal)

Pas de fichier de classification ici : les 4 grands comptes (Compte des transactions
courantes, Compte de capital, Compte financier, Erreurs et omissions) sont reperes par
leur intitule exact, et les postes de detail (Biens, Services, Revenu primaire, Revenu
secondaire, Investissements directs, Investissements de portefeuille) par une recherche
du libelle exact **a l'interieur de la section correspondante** -- pas par profondeur
d'indentation, qui n'est pas fiable d'un fichier a l'autre (les fichiers alternatifs RCA
n'ont plus d'indentation apres nettoyage en amont par leurs auteurs).

In [5]:
# Prefixe suffisamment long pour identifier chaque section sans ambiguite, tolerant aux
# fautes de frappe rencontrees dans certains fichiers (ex. un fichier RCA orthographie
# "ERREURS ET OMMISSIONS" avec deux M) -- match par prefixe plutot que chaine exacte.
SECTIONS_NIVEAU1 = {
    'COMPTE DES TRANSACTIONS COURANTES': 'COMPTE DES TRANSACTIONS COURANTES',
    'COMPTE DE CAPITAL': 'COMPTE DE CAPITAL',
    'COMPTE FINANCIER': 'COMPTE FINANCIER',
    'ERREURS ET OMISSIONS': 'ERREURS ET OM',
}


def _decouper_sections(df_pays_annee):
    """
    Renvoie {nom_section: (debut, fin)} -- bornes de lignes (index du DataFrame,
    exclusives en fin) entre deux en-tetes de section de niveau 1, reperes par le
    prefixe de SECTIONS_NIVEAU1 sur le libelle nettoye (en majuscules). Fonctionne que
    le fichier ait ou non des indentations (les en-tetes de section sont toujours en
    toutes lettres, avec ou sans espaces de tete).
    """
    libelles_maj = df_pays_annee['LIBELLE'].str.upper()
    positions = {}
    for section, prefixe in SECTIONS_NIVEAU1.items():
        idx = libelles_maj[libelles_maj.str.startswith(prefixe)].index
        if len(idx):
            positions[section] = idx[0]

    bornes = {}
    triees = sorted(positions.items(), key=lambda kv: kv[1])
    for i, (section, debut) in enumerate(triees):
        fin = triees[i + 1][1] if i + 1 < len(triees) else len(df_pays_annee)
        bornes[section] = (debut, fin)
    return bornes, positions


def _chercher_ligne(df_pays_annee, debut, fin, libelle_cible):
    """Premiere ligne dont le libelle nettoye correspond exactement (insensible aux
    espaces de tete) a libelle_cible, dans le segment [debut, fin[."""
    segment = df_pays_annee.iloc[debut:fin]
    match = segment[segment['LIBELLE'].str.strip() == libelle_cible]
    if match.empty:
        return None, None, None
    r = match.iloc[0]
    return r['CREDIT_USD_M'], r['DEBIT_USD_M'], r['SOLDE_USD_M']


def extraire_agregats_pays_annee(df_pays_annee):
    """
    Reconstitue, a partir des lignes brutes d'un (pays, annee), les agregats necessaires
    aux 10 metriques -- meme logique que BOP_PAR_COMPTE.csv (section 14 du notebook
    principal) mais sans fichier de classification : les 4 grands comptes BPM6 sont
    reperes par leur intitule exact (SECTIONS_NIVEAU1), les postes de detail par
    recherche du libelle exact a l'interieur de la section correspondante (pas par
    profondeur d'indentation, non fiable d'un fichier a l'autre).
    """
    bornes, positions = _decouper_sections(df_pays_annee)

    resultat = {
        'CREDIT_BIENS': None, 'DEBIT_BIENS': None,
        'CREDIT_SERVICES': None, 'DEBIT_SERVICES': None,
        'CREDIT_REVENU_PRIMAIRE': None, 'DEBIT_REVENU_PRIMAIRE': None,
        'CREDIT_REVENU_SECONDAIRE': None, 'DEBIT_REVENU_SECONDAIRE': None,
        'SOLDE_COURANT': None, 'COMPTE_CAPITAL': None,
        'IDE_NET': None, 'PORTEFEUILLE_NET': None, 'EO_NET': None,
    }

    if 'COMPTE DES TRANSACTIONS COURANTES' in bornes:
        debut, fin = bornes['COMPTE DES TRANSACTIONS COURANTES']
        pos = positions['COMPTE DES TRANSACTIONS COURANTES']
        resultat['CREDIT_BIENS'], resultat['DEBIT_BIENS'], _ = _chercher_ligne(df_pays_annee, debut, fin, 'Biens')
        _, _, sc = _chercher_ligne(df_pays_annee, pos, pos + 1, 'COMPTE DES TRANSACTIONS COURANTES')
        c_sc, d_sc, s_sc = df_pays_annee.iloc[pos][['CREDIT_USD_M', 'DEBIT_USD_M', 'SOLDE_USD_M']]
        resultat['SOLDE_COURANT'] = s_sc
        resultat['CREDIT_SERVICES'], resultat['DEBIT_SERVICES'], _ = _chercher_ligne(df_pays_annee, debut, fin, 'Services')
        resultat['CREDIT_REVENU_PRIMAIRE'], resultat['DEBIT_REVENU_PRIMAIRE'], _ = _chercher_ligne(df_pays_annee, debut, fin, 'Revenu primaire')
        resultat['CREDIT_REVENU_SECONDAIRE'], resultat['DEBIT_REVENU_SECONDAIRE'], _ = _chercher_ligne(df_pays_annee, debut, fin, 'Revenu secondaire')

    if 'COMPTE DE CAPITAL' in positions:
        pos = positions['COMPTE DE CAPITAL']
        resultat['COMPTE_CAPITAL'] = df_pays_annee.iloc[pos]['SOLDE_USD_M']

    if 'COMPTE FINANCIER' in bornes:
        debut, fin = bornes['COMPTE FINANCIER']
        _, _, resultat['IDE_NET'] = _chercher_ligne(df_pays_annee, debut, fin, 'Investissements directs')
        _, _, resultat['PORTEFEUILLE_NET'] = _chercher_ligne(df_pays_annee, debut, fin, 'Investissements de portefeuille')

    if 'ERREURS ET OMISSIONS' in positions:
        pos = positions['ERREURS ET OMISSIONS']
        resultat['EO_NET'] = df_pays_annee.iloc[pos]['SOLDE_USD_M']

    return resultat


lignes_agregats = []
for (code, annee), df_pays_annee in df_brut_cemac.groupby(['COUNTRY_CODE', 'ANNEE']):
    df_pays_annee = df_pays_annee.reset_index(drop=True)
    agg = extraire_agregats_pays_annee(df_pays_annee)
    agg['COUNTRY_CODE'] = code
    agg['ANNEE'] = annee
    lignes_agregats.append(agg)

df_agregats_cemac = pd.DataFrame(lignes_agregats)
colonnes_ordre = ['COUNTRY_CODE', 'ANNEE'] + [c for c in df_agregats_cemac.columns if c not in ('COUNTRY_CODE', 'ANNEE')]
df_agregats_cemac = df_agregats_cemac[colonnes_ordre]

print(f"Agregats extraits pour {len(df_agregats_cemac)} couples (pays, annee) -- valeurs en millions USD "
      f"(convertis depuis millions FCFA, taux fixe {TAUX_XAF_USD}).")
display(df_agregats_cemac)


Agregats extraits pour 22 couples (pays, annee) -- valeurs en millions USD (convertis depuis millions FCFA, taux fixe 0.001771).


,COUNTRY_CODE,ANNEE,CREDIT_BIENS,DEBIT_BIENS,CREDIT_SERVICES,DEBIT_SERVICES,CREDIT_REVENU_PRIMAIRE,DEBIT_REVENU_PRIMAIRE,CREDIT_REVENU_SECONDAIRE,DEBIT_REVENU_SECONDAIRE,SOLDE_COURANT,COMPTE_CAPITAL,IDE_NET,PORTEFEUILLE_NET,EO_NET
0,CAF,2020,136.583062,515.334435,296.738134,361.230870,88.743039,33.923505,311.409098,86.086539,-163.102016,150.241014,-8.644251,-5.181946,25.010062
1,CAF,2021,147.122283,450.205910,265.773970,290.865498,82.755288,7.928767,186.895401,89.890647,-156.343880,69.582590,22.705991,14.835667,-44.460955
2,CAF,2023,183.893556,467.544000,194.034302,264.381964,61.131378,0.309925,190.680028,8.029714,-110.526339,70.367143,31.757572,-14.757743,15.462601
3,CMR,2020,4494.710326,5187.749063,1727.333381,2146.005087,NaN,NaN,NaN,NaN,-1544.274661,74.840866,604.378544,33.365599,13.607011
4,CMR,2021,5769.096846,6460.410070,1554.247906,2414.785045,NaN,NaN,NaN,NaN,-1765.386122,118.893251,891.957066,-177.075560,-22.860068
5,CMR,2022,7654.313788,8003.680677,1900.014897,2800.557841,NaN,NaN,NaN,NaN,-1668.682106,238.587562,992.355392,52.795281,43.133150
6,COG,2020,4521.711887,2280.621189,238.651105,1061.482499,27.381431,91.355264,150.802421,36.562295,1468.525597,26.566771,1021.381746,-137.654517,202.469575
7,COG,2021,7376.923400,2729.560834,235.654573,1676.676540,59.339126,1680.413350,146.206676,46.325818,1685.147233,24.390212,324.259474,-64.921318,-355.168737
8,COG,2022,10498.247144,3224.344585,235.711245,1793.241989,99.767514,2031.213030,276.858659,47.328204,4014.456754,3.302915,1378.948417,-394.074065,693.105644
9,COG,2023,8240.275274,4919.356288,325.272486,2947.514262,456.333570,2141.817293,298.597684,62.211688,-750.420517,1.020096,-1320.356653,-819.110523,143.893750


---
## 5. PIB par secteur (prix courants, `deor_<pays>.xlsx`)

In [6]:
def _sans_accents(texte):
    return ''.join(c for c in unicodedata.normalize('NFD', texte) if unicodedata.category(c) != 'Mn')


def _est_ligne_pib_total(label):
    """
    Reconnait la ligne de PIB nominal total (pas un sous-secteur) malgre les variantes
    de formulation observees d'un fichier pays a l'autre : "PIB aux prix courants du
    marche" pour la plupart des pays, "PIB AU PRIX COURANT DU MARCHE" (singulier, tout
    majuscule) pour la RCA -- match insensible aux accents/casse/pluriel plutot
    qu'une chaine exacte.
    """
    n = _sans_accents(_normaliser(label)).upper()
    return 'PIB' in n and 'PRIX COURANT' in n and 'MARCHE' in n


def lire_pib_secteur(chemin, annees_voulues):
    """
    Lit un fichier deor_<pays>.xlsx (PIB par secteur, prix courants, milliards de FCFA) :
    trouve la ligne du PIB nominal total (pas un sous-secteur) et la ligne des annees en
    en-tete, retourne {annee: PIB_milliards_FCFA} pour les annees demandees.
    """
    wb = openpyxl.load_workbook(chemin, data_only=True)
    ws = wb[wb.sheetnames[0]]

    ligne_annees = None
    ligne_pib = None
    for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
        if ligne_pib is None and _est_ligne_pib_total(row[0]):
            ligne_pib = row
        if ligne_annees is None:
            annees_ligne = [v for v in row if isinstance(v, (int, float)) and 1990 <= v <= 2100]
            if len(annees_ligne) >= 5:
                ligne_annees = row

    if ligne_annees is None or ligne_pib is None:
        raise ValueError(f"Ligne PIB ou ligne d'annees introuvable dans {chemin.name}")

    resultat = {}
    for col, val in enumerate(ligne_annees):
        if isinstance(val, (int, float)) and int(val) in annees_voulues:
            pib_valeur = ligne_pib[col] if col < len(ligne_pib) else None
            resultat[int(val)] = pd.to_numeric(pib_valeur, errors='coerce')
    return resultat


lignes_pib = []
for code, nom_fichier in FICHIERS_PIB_CEMAC.items():
    chemin = DIR_PIB_CEMAC / nom_fichier
    pib_par_annee = lire_pib_secteur(chemin, ANNEES_ETUDE)
    for annee, pib_md_fcfa in pib_par_annee.items():
        lignes_pib.append({
            'COUNTRY_CODE': code,
            'ANNEE': annee,
            'PIB_MILLIARDS_FCFA': pib_md_fcfa,
            'PIB_USD_MILLIONS': pib_md_fcfa * 1000 * TAUX_XAF_USD if pd.notna(pib_md_fcfa) else None,
        })

df_pib_cemac = pd.DataFrame(lignes_pib)
chemin_pib_csv = DIR_SORTIE / 'PIB_CEMAC_BEAC.csv'
df_pib_cemac.to_csv(chemin_pib_csv, index=False)
print(f"{chemin_pib_csv} : {len(df_pib_cemac)} lignes (pays x annee).")
display(df_pib_cemac)


m:\Travail\BEAC\balance_de_paiement\data\PIB_CEMAC_BEAC.csv : 24 lignes (pays x annee).


,COUNTRY_CODE,ANNEE,PIB_MILLIARDS_FCFA,PIB_USD_MILLIONS
0,COG,2020,6400.858430,11335.920279
1,COG,2021,8323.559929,14741.024635
2,COG,2022,10400.314750,18418.957422
3,COG,2023,9586.306479,16977.348775
4,GAB,2020,8398.890237,14874.434609
5,GAB,2021,9994.278433,17699.867105
6,GAB,2022,12131.445547,21484.790063
7,GAB,2023,11471.431886,20315.905870
8,GNQ,2020,5782.405691,10240.640479
9,GNQ,2021,6937.719636,12286.701475


---
## 6. Metriques d'analyse externe (memes 10 formules que le notebook principal,
section 15)

**Limite documentee** : la Balance de base utilise ici tout le compte "Investissements
de portefeuille" (pas de distinction Long terme / Court terme disponible dans ces
fichiers BEAC, contrairement a `RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv`
utilise cote FMI) -- approximation, pas une donnee inventee.

In [7]:
metriques_cemac_local = df_agregats_cemac.merge(df_pib_cemac[['COUNTRY_CODE', 'ANNEE', 'PIB_USD_MILLIONS']],
                                                 on=['COUNTRY_CODE', 'ANNEE'], how='left')

metriques_cemac_local['TC_TAUX_COUVERTURE_BIENS'] = metriques_cemac_local['CREDIT_BIENS'] / metriques_cemac_local['DEBIT_BIENS'] * 100
metriques_cemac_local['TM_TAUX_COUVERTURE_IMPORTATIONS'] = metriques_cemac_local['DEBIT_BIENS'] / metriques_cemac_local['CREDIT_BIENS'] * 100

# TAS/TFR ne comptent que le DEFICIT net (cf. notebook principal, section 15) : un
# surplus net de services ou de revenu primaire n'entre pas en fuite.
metriques_cemac_local['TAS_TAUX_ABSORPTION_SERVICES'] = (
    metriques_cemac_local['DEBIT_SERVICES'] - metriques_cemac_local['CREDIT_SERVICES']
).clip(lower=0) / metriques_cemac_local['CREDIT_BIENS'] * 100
metriques_cemac_local['TFR_TAUX_FUITE_REVENU'] = (
    metriques_cemac_local['DEBIT_REVENU_PRIMAIRE'] - metriques_cemac_local['CREDIT_REVENU_PRIMAIRE']
).clip(lower=0) / metriques_cemac_local['CREDIT_BIENS'] * 100
metriques_cemac_local['TRE_TAUX_RETENTION'] = (
    100 - metriques_cemac_local['TM_TAUX_COUVERTURE_IMPORTATIONS']
    - metriques_cemac_local['TAS_TAUX_ABSORPTION_SERVICES'] - metriques_cemac_local['TFR_TAUX_FUITE_REVENU']
)

metriques_cemac_local['RATIO_IDE_SOLDE_COURANT'] = metriques_cemac_local['IDE_NET'] / metriques_cemac_local['SOLDE_COURANT'].abs() * 100
metriques_cemac_local['RATIO_ERREURS_OMISSIONS'] = metriques_cemac_local['EO_NET'].abs() / metriques_cemac_local['PIB_USD_MILLIONS'] * 100
metriques_cemac_local['SOLDE_COURANT_PIB'] = metriques_cemac_local['SOLDE_COURANT'] / metriques_cemac_local['PIB_USD_MILLIONS'] * 100

# Balance de base : le detail Long terme / Court terme du portefeuille n'est pas
# disponible dans ces fichiers BEAC (pas de colonne MATURITE comme dans
# RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv, section 14 du notebook
# principal) -- on utilise tout le compte "Investissements de portefeuille" comme
# approximation. Limite documentee, pas une donnee inventee.
metriques_cemac_local['BALANCE_DE_BASE'] = (
    metriques_cemac_local['SOLDE_COURANT'] + metriques_cemac_local['IDE_NET']
    + metriques_cemac_local['PORTEFEUILLE_NET'].fillna(0)
)

metriques_cemac_local['TAUX_OUVERTURE_COMMERCIALE'] = (
    metriques_cemac_local['CREDIT_BIENS'] + metriques_cemac_local['CREDIT_SERVICES']
    + metriques_cemac_local['DEBIT_BIENS'] + metriques_cemac_local['DEBIT_SERVICES']
) / metriques_cemac_local['PIB_USD_MILLIONS'] * 100

chemin_metriques_local = DIR_SORTIE / 'METRIQUES_CEMAC_BEAC.csv'
metriques_cemac_local.to_csv(chemin_metriques_local, index=False)
print(f"{chemin_metriques_local} : {len(metriques_cemac_local)} lignes, "
      f"{metriques_cemac_local['COUNTRY_CODE'].nunique()} pays -- valeurs en millions USD.")
display(metriques_cemac_local)


m:\Travail\BEAC\balance_de_paiement\data\METRIQUES_CEMAC_BEAC.csv : 22 lignes, 6 pays -- valeurs en millions USD.


,COUNTRY_CODE,ANNEE,CREDIT_BIENS,DEBIT_BIENS,CREDIT_SERVICES,DEBIT_SERVICES,CREDIT_REVENU_PRIMAIRE,DEBIT_REVENU_PRIMAIRE,CREDIT_REVENU_SECONDAIRE,DEBIT_REVENU_SECONDAIRE,...,TC_TAUX_COUVERTURE_BIENS,TM_TAUX_COUVERTURE_IMPORTATIONS,TAS_TAUX_ABSORPTION_SERVICES,TFR_TAUX_FUITE_REVENU,TRE_TAUX_RETENTION,RATIO_IDE_SOLDE_COURANT,RATIO_ERREURS_OMISSIONS,SOLDE_COURANT_PIB,BALANCE_DE_BASE,TAUX_OUVERTURE_COMMERCIALE
0,CAF,2020,136.583062,515.334435,296.738134,361.230870,88.743039,33.923505,311.409098,86.086539,...,26.503772,377.304790,47.218692,0.000000,-324.523482,-5.299904,1.028601,-6.707979,-176.928213,53.872361
1,CAF,2021,147.122283,450.205910,265.773970,290.865498,82.755288,7.928767,186.895401,89.890647,...,32.678888,306.007969,17.054879,0.000000,-223.062848,14.523108,1.796549,-6.317442,-118.802222,46.628775
2,CAF,2023,183.893556,467.544000,194.034302,264.381964,61.131378,0.309925,190.680028,8.029714,...,39.331818,254.247082,38.254555,0.000000,-192.501637,28.733035,0.599605,-4.285965,-93.526510,43.037657
3,CMR,2020,4494.710326,5187.749063,1727.333381,2146.005087,NaN,NaN,NaN,NaN,...,86.640859,115.418986,9.314765,NaN,NaN,39.136726,0.033281,-3.777133,-906.530517,33.156049
4,CMR,2021,5769.096846,6460.410070,1554.247906,2414.785045,NaN,NaN,NaN,NaN,...,89.299236,111.983041,14.916323,NaN,NaN,50.524758,0.054968,-4.244927,-1050.504617,38.949900
5,CMR,2022,7654.313788,8003.680677,1900.014897,2800.557841,NaN,NaN,NaN,NaN,...,95.634922,104.564314,11.765169,NaN,NaN,59.469409,0.099648,-3.855064,-623.531433,47.033274
6,COG,2020,4521.711887,2280.621189,238.651105,1061.482499,27.381431,91.355264,150.802421,36.562295,...,198.266679,50.437119,18.197342,1.414814,29.950725,69.551511,1.786089,12.954622,2352.252826,71.476038
7,COG,2021,7376.923400,2729.560834,235.654573,1676.676540,59.339126,1680.413350,146.206676,46.325818,...,270.260450,37.001344,19.534186,21.974936,21.489533,19.242204,2.409390,11.431683,1944.485389,81.533107
8,COG,2022,10498.247144,3224.344585,235.711245,1793.241989,99.767514,2031.213030,276.858659,47.328204,...,325.593213,30.713171,14.836103,18.397791,36.052936,34.349565,3.763001,21.795244,4999.331106,85.518114
9,COG,2023,8240.275274,4919.356288,325.272486,2947.514262,456.333570,2141.817293,298.597684,62.211688,...,167.507186,59.698931,31.822259,20.454216,-11.975407,-175.948901,0.847563,-4.420128,-2889.887693,96.790250


---
## 7. Tableaux comparatifs (memes conventions que la section 16 du notebook principal)

Un CSV complet par type de tableau (tous pays x toutes annees disponibles), et juste un
exemple illustratif par pays + un exemple comparatif par annee affiches/exportes en PNG --
pas une figure par pays/annee.

In [8]:
ANNEES_PAR_PAYS_LOCAL = {
    code: sorted(metriques_cemac_local[metriques_cemac_local['COUNTRY_CODE'] == code]['ANNEE'].unique())
    for code in PAYS_CEMAC_LOCAL
    if code in metriques_cemac_local['COUNTRY_CODE'].unique()
}
ANNEES_UNION_LOCAL = sorted(set.union(*(set(v) for v in ANNEES_PAR_PAYS_LOCAL.values())))

print("Annees disponibles par pays (source BEAC locale) :")
for code, annees in ANNEES_PAR_PAYS_LOCAL.items():
    print(f"  {PAYS_CEMAC_LOCAL[code]:<28} ({code}) : {annees}")


def ligne_pour(code, annee):
    sub = metriques_cemac_local[(metriques_cemac_local['COUNTRY_CODE'] == code) & (metriques_cemac_local['ANNEE'] == annee)]
    if sub.empty:
        return None
    return sub.iloc[0]


def fmt_local(v, mode='musd'):
    """'musd' -> entier signe + espace fine, 'pct' -> 1 decimale, 'ratio' -> 2 decimales virgule."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'n.d.'
    NB = ' '
    MIN = '−'
    if mode == 'pct':
        return f'+{v:.1f}{NB}%' if v >= 0 else f'{MIN}{abs(v):.1f}{NB}%'
    if mode == 'ratio':
        return f'{v:.2f}'.replace('.', ',')
    v = int(round(v))
    sign = '+' if v >= 0 else ''
    num = f'{abs(v):,}'.replace(',', NB)
    return f"{sign if v >= 0 else MIN}{num}"


def render_table_local(df, title, caption, highlight_rows=None, filename=None, fontsize=13):
    """Meme rendu Plotly que le notebook principal (section 16, render_table_cemac)."""
    HEADER_BG = '#00448C'
    HL_COLOR = '#D6E4F0'
    ALT_COLOR = '#F4F7FB'

    row_labels = list(df.index)
    col_labels = list(df.columns)
    n_rows = len(row_labels)
    highlight_rows = set(highlight_rows or [])

    def cellule(i, texte):
        return f'<b>{texte}</b>' if i in highlight_rows else texte

    header_vals = ['<b></b>'] + [f'<b>{c}</b>' for c in col_labels]
    cell_vals = [[cellule(i, lbl) for i, lbl in enumerate(row_labels)]]
    for c in col_labels:
        cell_vals.append([cellule(i, v) for i, v in enumerate(df[c].tolist())])

    fill_colors = []
    for _ in range(len(col_labels) + 1):
        col_fill = [HL_COLOR if i in highlight_rows else (ALT_COLOR if i % 2 == 0 else 'white') for i in range(n_rows)]
        fill_colors.append(col_fill)

    fig = go.Figure(data=[go.Table(
        columnwidth=[2.6] + [1.6] * len(col_labels),
        header=dict(
            values=header_vals, fill_color=HEADER_BG,
            font=dict(color='white', size=fontsize, family='Arial'),
            align=['left'] + ['center'] * len(col_labels), height=34,
        ),
        cells=dict(
            values=cell_vals, fill_color=fill_colors,
            font=dict(color='#1a1a1a', size=fontsize - 1, family='Arial'),
            align=['left'] + ['center'] * len(col_labels), height=28,
            line_color='#C8D0DB',
        ),
    )])

    fig.update_layout(
        title=dict(text=title.replace('\n', '<br>'), x=0.01, xanchor='left',
                    font=dict(size=16, family='Arial', color='#1a1a1a')),
        margin=dict(l=10, r=10, t=80, b=50),
        width=max(760, 210 * len(col_labels) + 320),
        height=max(280, n_rows * 30 + 150),
        paper_bgcolor='white',
        annotations=[dict(
            text=caption.replace('\n', '<br>'), xref='paper', yref='paper', x=0, y=-0.05,
            showarrow=False, font=dict(size=10, color='#555555'), align='left',
        )],
    )

    if filename:
        fig.write_image(str(filename), scale=2)
    return fig


Annees disponibles par pays (source BEAC locale) :
  Congo                        (COG) : [2020, 2021, 2022, 2023]
  Gabon                        (GAB) : [2020, 2021, 2022, 2023]
  Guinée équatoriale           (GNQ) : [2020, 2021, 2022, 2023]
  Tchad                        (TCD) : [2020, 2021, 2022, 2023]
  République centrafricaine    (CAF) : [2020, 2021, 2023]
  Cameroun                     (CMR) : [2020, 2021, 2022]


In [9]:
rows_t1_local = [
    ('Compte courant', 'solde_courant'),
    ('  Biens', 'biens'),
    ('  Services', 'services'),
    ('  Revenu primaire', 'rev_primaire'),
    ('  Revenu secondaire', 'rev_secondaire'),
    ('Compte de capital', 'compte_capital'),
    ('Capacité de financement', 'capacite_finan'),
    ('  dont IDE net', 'ide_net'),
    ('  dont Portefeuille net', 'portefeuille_net'),
    ('Erreurs et omissions', 'eo_net'),
    ('PIB (M USD)', 'pib'),
]


def valeurs_t1_local(code, annee):
    r = ligne_pour(code, annee)
    if r is None:
        return {cle: None for _, cle in rows_t1_local}
    return {
        'solde_courant': r['SOLDE_COURANT'],
        'biens': r['CREDIT_BIENS'] - r['DEBIT_BIENS'],
        'services': r['CREDIT_SERVICES'] - r['DEBIT_SERVICES'],
        'rev_primaire': r['CREDIT_REVENU_PRIMAIRE'] - r['DEBIT_REVENU_PRIMAIRE'],
        'rev_secondaire': r['CREDIT_REVENU_SECONDAIRE'] - r['DEBIT_REVENU_SECONDAIRE'],
        'compte_capital': r['COMPTE_CAPITAL'],
        'capacite_finan': r['SOLDE_COURANT'] + r['COMPTE_CAPITAL'],
        'ide_net': r['IDE_NET'],
        'portefeuille_net': r['PORTEFEUILLE_NET'],
        'eo_net': r['EO_NET'],
        'pib': r['PIB_USD_MILLIONS'],
    }


def colonne_t1_local(code, annee):
    vals = valeurs_t1_local(code, annee)
    return [fmt_local(vals[cle]) for _, cle in rows_t1_local]


# Tableau 1 complet (tous pays x toutes leurs annees) -> CSV ---
lignes_t1_local = []
for code, nom in PAYS_CEMAC_LOCAL.items():
    if code not in ANNEES_PAR_PAYS_LOCAL:
        continue
    for a in ANNEES_PAR_PAYS_LOCAL[code]:
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        ligne.update(valeurs_t1_local(code, a))
        lignes_t1_local.append(ligne)

df_t1_local_complet = pd.DataFrame(lignes_t1_local)
chemin_t1_local_csv = DIR_SORTIE / 'TABLEAU1_SOLDES_CEMAC_BEAC.csv'
df_t1_local_complet.to_csv(chemin_t1_local_csv, index=False)
print(f"{chemin_t1_local_csv} : {len(df_t1_local_complet)} lignes ({df_t1_local_complet['COUNTRY_CODE'].nunique()} pays x annees), "
      "valeurs en millions USD (source : releves BEAC, taux fixe XAF/USD).")

# Exemple par pays ---
code_ex = next(iter(ANNEES_PAR_PAYS_LOCAL))
nom_ex = PAYS_CEMAC_LOCAL[code_ex]
annees_ex = ANNEES_PAR_PAYS_LOCAL[code_ex]
data_ex = {a: colonne_t1_local(code_ex, a) for a in annees_ex}
df_t1_ex = pd.DataFrame(data_ex, index=[r[0] for r in rows_t1_local])

fig_t1_ex = render_table_local(
    df_t1_ex,
    title=f'Tableau 1 (exemple)  {nom_ex}\nPrincipaux soldes de la balance des paiements '
          f'({annees_ex[0]}–{annees_ex[-1]}, millions USD)',
    caption=f"Exemple illustratif parmi {len(ANNEES_PAR_PAYS_LOCAL)} pays -- serie complete dans "
            f"{chemin_t1_local_csv.name}. Source : releves BEAC (Balance des Paiements analytique), "
            "conversion FCFA->USD a taux fixe.",
    highlight_rows=[0, 5, 6],
    filename=FIGURES_DIR / f'tab1_soldes_beac_{code_ex}.png',
)
fig_t1_ex.show()

# Exemple comparatif ---
annees_comparables_t1_local = [a for a in ANNEES_UNION_LOCAL if len([c for c in ANNEES_PAR_PAYS_LOCAL if a in ANNEES_PAR_PAYS_LOCAL[c]]) >= 2]
if annees_comparables_t1_local:
    annee_ex1 = annees_comparables_t1_local[-1]
    pays_dispo_ex1 = [c for c in ANNEES_PAR_PAYS_LOCAL if annee_ex1 in ANNEES_PAR_PAYS_LOCAL[c]]
    data_comp1 = {PAYS_CEMAC_LOCAL[c]: colonne_t1_local(c, annee_ex1) for c in pays_dispo_ex1}
    df_t1_comp_ex1 = pd.DataFrame(data_comp1, index=[r[0] for r in rows_t1_local])

    fig_t1_comp_ex1 = render_table_local(
        df_t1_comp_ex1,
        title=f'Tableau 1 comparatif (exemple)  {annee_ex1}\nPrincipaux soldes de la balance des paiements (millions USD)',
        caption=f"Exemple illustratif -- {len(pays_dispo_ex1)} pays disponibles cette annee-la sur "
                f"{len(ANNEES_PAR_PAYS_LOCAL)} au total ; serie complete dans {chemin_t1_local_csv.name}.",
        highlight_rows=[0, 5, 6],
        filename=FIGURES_DIR / f'tab1_soldes_beac_comparatif_{annee_ex1}.png',
    )
    fig_t1_comp_ex1.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU1_SOLDES_CEMAC_BEAC.csv : 22 lignes (6 pays x annees), valeurs en millions USD (source : releves BEAC, taux fixe XAF/USD).


In [10]:
rows_t2_local = [
    ('Exportations de biens (M USD)', 'exp_biens', 'musd'),
    ('Importations de biens (M USD)', 'imp_biens', 'musd'),
    ('Taux de couverture (TC)', 'tc', 'ratio'),
    ("Taux de couverture des importations (TM)", 'tm', 'ratio'),
    ("Taux d'absorption services (TAS)", 'tas', 'pct_simple'),
    ('Taux de fuite revenu primaire (TFR)', 'tfr', 'pct_simple'),
    ('Taux de rétention (TRE)', 'tre', 'pct_simple'),
]


def formatter_t2_local(vals, cle, mode):
    v = vals.get(cle)
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'n.d.'
    if mode == 'musd':
        return fmt_local(v, 'musd')
    if mode == 'ratio':
        return f"{v:.2f}".replace('.', ',')
    return f"{v:.1f} %"


def valeurs_t2_local(code, annee):
    r = ligne_pour(code, annee)
    if r is None:
        return {}
    return {
        'exp_biens': r['CREDIT_BIENS'], 'imp_biens': r['DEBIT_BIENS'],
        'tc': r['TC_TAUX_COUVERTURE_BIENS'], 'tm': r['TM_TAUX_COUVERTURE_IMPORTATIONS'],
        'tas': r['TAS_TAUX_ABSORPTION_SERVICES'], 'tfr': r['TFR_TAUX_FUITE_REVENU'],
        'tre': r['TRE_TAUX_RETENTION'],
    }


def colonne_t2_local(code, annee):
    vals = valeurs_t2_local(code, annee)
    return [formatter_t2_local(vals, cle, mode) for _, cle, mode in rows_t2_local]


# Tableau 2 complet -> CSV ---
lignes_t2_local = []
for code, nom in PAYS_CEMAC_LOCAL.items():
    if code not in ANNEES_PAR_PAYS_LOCAL:
        continue
    for a in ANNEES_PAR_PAYS_LOCAL[code]:
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        ligne.update(valeurs_t2_local(code, a))
        lignes_t2_local.append(ligne)

df_t2_local_complet = pd.DataFrame(lignes_t2_local)
chemin_t2_local_csv = DIR_SORTIE / 'TABLEAU2_TERMES_ECHANGE_CEMAC_BEAC.csv'
df_t2_local_complet.to_csv(chemin_t2_local_csv, index=False)
print(f"{chemin_t2_local_csv} : {len(df_t2_local_complet)} lignes ({df_t2_local_complet['COUNTRY_CODE'].nunique()} pays x annees).")

# Exemple par pays ---
code_ex2 = next(iter(ANNEES_PAR_PAYS_LOCAL))
nom_ex2 = PAYS_CEMAC_LOCAL[code_ex2]
annees_ex2 = ANNEES_PAR_PAYS_LOCAL[code_ex2]
data_ex2 = {a: colonne_t2_local(code_ex2, a) for a in annees_ex2}
df_t2_ex2 = pd.DataFrame(data_ex2, index=[r[0] for r in rows_t2_local])

fig_t2_ex2 = render_table_local(
    df_t2_ex2,
    title=f"Tableau 2 (exemple)  {nom_ex2}\nIndicateurs de termes de l'échange ({annees_ex2[0]}–{annees_ex2[-1]})",
    caption=f"Exemple illustratif parmi {len(ANNEES_PAR_PAYS_LOCAL)} pays -- serie complete dans {chemin_t2_local_csv.name}.",
    highlight_rows=[2, 3, 6],
    filename=FIGURES_DIR / f'tab2_termes_echange_beac_{code_ex2}.png',
)
fig_t2_ex2.show()

# Exemple comparatif ---
annees_comparables_t2_local = [a for a in ANNEES_UNION_LOCAL if len([c for c in ANNEES_PAR_PAYS_LOCAL if a in ANNEES_PAR_PAYS_LOCAL[c]]) >= 2]
if annees_comparables_t2_local:
    annee_ex2 = annees_comparables_t2_local[-3]
    pays_dispo_ex2 = [c for c in ANNEES_PAR_PAYS_LOCAL if annee_ex2 in ANNEES_PAR_PAYS_LOCAL[c]]
    data_comp2 = {PAYS_CEMAC_LOCAL[c]: colonne_t2_local(c, annee_ex2) for c in pays_dispo_ex2}
    df_t2_comp_ex2 = pd.DataFrame(data_comp2, index=[r[0] for r in rows_t2_local])

    fig_t2_comp_ex2 = render_table_local(
        df_t2_comp_ex2,
        title=f"Tableau 2 comparatif (exemple)  {annee_ex2}\nIndicateurs de termes de l'échange",
        caption=f"Exemple illustratif -- {len(pays_dispo_ex2)} pays disponibles cette annee-la sur "
                f"{len(ANNEES_PAR_PAYS_LOCAL)} au total ; serie complete dans {chemin_t2_local_csv.name}.",
        highlight_rows=[2, 3, 6],
        filename=FIGURES_DIR / f'tab2_termes_echange_beac_comparatif_{annee_ex2}.png',
    )
    fig_t2_comp_ex2.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU2_TERMES_ECHANGE_CEMAC_BEAC.csv : 22 lignes (6 pays x annees).


In [11]:
def qualite_local(code, annee):
    r = ligne_pour(code, annee)
    if r is None or pd.isna(r['SOLDE_COURANT']) or r['SOLDE_COURANT'] == 0:
        return 'n.d.'
    ratio = abs(r['EO_NET']) / abs(r['SOLDE_COURANT']) * 100
    if ratio < 10:
        return 'Bonne'
    if ratio < 60:
        return 'Moyenne'
    return 'Faible'


rows_t3_local = [
    'Solde courant (M USD)',
    'Solde courant / PIB',
    'Taux de couverture biens (TC)',
    "Taux de couverture importations (TM)",
    "Taux d'absorption services (TAS)",
    'Taux de fuite revenu primaire (TFR)',
    'Taux de rétention (TRE)',
    'Balance de base (M USD)',
    'Erreurs & Omissions (M USD)',
    'Qualité statistique',
]


def colonne_t3_local(code, annee):
    r = ligne_pour(code, annee)
    if r is None:
        return ['n.d.'] * len(rows_t3_local)
    return [
        fmt_local(r['SOLDE_COURANT']),
        fmt_local(r['SOLDE_COURANT_PIB'], 'pct') if pd.notna(r['SOLDE_COURANT_PIB']) else 'n.d.',
        f"{r['TC_TAUX_COUVERTURE_BIENS']:.2f}".replace('.', ',') if pd.notna(r['TC_TAUX_COUVERTURE_BIENS']) else 'n.d.',
        f"{r['TM_TAUX_COUVERTURE_IMPORTATIONS']:.2f}".replace('.', ',') if pd.notna(r['TM_TAUX_COUVERTURE_IMPORTATIONS']) else 'n.d.',
        f"{r['TAS_TAUX_ABSORPTION_SERVICES']:.1f} %" if pd.notna(r['TAS_TAUX_ABSORPTION_SERVICES']) else 'n.d.',
        f"{r['TFR_TAUX_FUITE_REVENU']:.1f} %" if pd.notna(r['TFR_TAUX_FUITE_REVENU']) else 'n.d.',
        f"{r['TRE_TAUX_RETENTION']:.1f} %" if pd.notna(r['TRE_TAUX_RETENTION']) else 'n.d.',
        fmt_local(r['BALANCE_DE_BASE']),
        fmt_local(r['EO_NET']),
        qualite_local(code, annee),
    ]


# Tableau 3 complet -> CSV (reutilise directement metriques_cemac_local) ---
chemin_t3_local_csv = DIR_SORTIE / 'TABLEAU3_DIAGNOSTIC_CEMAC_BEAC.csv'
metriques_cemac_local.to_csv(chemin_t3_local_csv, index=False)
print(f"{chemin_t3_local_csv} : {len(metriques_cemac_local)} lignes ({metriques_cemac_local['COUNTRY_CODE'].nunique()} pays x annees).")

# Exemple par pays ---
code_ex3 = next(iter(ANNEES_PAR_PAYS_LOCAL))
nom_ex3 = PAYS_CEMAC_LOCAL[code_ex3]
annees_ex3 = ANNEES_PAR_PAYS_LOCAL[code_ex3]
data_ex3 = {a: colonne_t3_local(code_ex3, a) for a in annees_ex3}
df_t3_ex3 = pd.DataFrame(data_ex3, index=rows_t3_local)

fig_t3_ex3 = render_table_local(
    df_t3_ex3,
    title=f'Tableau 3 (exemple)  {nom_ex3}\nTableau de bord comparatif (diagnostic macroéconomique), '
          f'{annees_ex3[0]}–{annees_ex3[-1]}',
    caption=f"Exemple illustratif parmi {len(ANNEES_PAR_PAYS_LOCAL)} pays -- serie complete dans {chemin_t3_local_csv.name}.",
    highlight_rows=[0, 6],
    filename=FIGURES_DIR / f'tab3_diagnostic_beac_{code_ex3}.png',
)
fig_t3_ex3.show()

# Exemple comparatif ---
annees_comparables_t3_local = [a for a in ANNEES_UNION_LOCAL if len([c for c in ANNEES_PAR_PAYS_LOCAL if a in ANNEES_PAR_PAYS_LOCAL[c]]) >= 2]
if annees_comparables_t3_local:
    annee_ex3 = annees_comparables_t3_local[-2]
    pays_dispo_ex3 = [c for c in ANNEES_PAR_PAYS_LOCAL if annee_ex3 in ANNEES_PAR_PAYS_LOCAL[c]]
    data_comp3 = {PAYS_CEMAC_LOCAL[c]: colonne_t3_local(c, annee_ex3) for c in pays_dispo_ex3}
    df_t3_comp_ex3 = pd.DataFrame(data_comp3, index=rows_t3_local)

    fig_t3_comp_ex3 = render_table_local(
        df_t3_comp_ex3,
        title=f'Tableau 3 comparatif (exemple)  {annee_ex3}\nTableau de bord comparatif (diagnostic macroéconomique)',
        caption=f"Exemple illustratif -- {len(pays_dispo_ex3)} pays disponibles cette annee-la sur "
                f"{len(ANNEES_PAR_PAYS_LOCAL)} au total ; serie complete dans {chemin_t3_local_csv.name}.",
        highlight_rows=[0, 6],
        filename=FIGURES_DIR / f'tab3_diagnostic_beac_comparatif_{annee_ex3}.png',
    )
    fig_t3_comp_ex3.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU3_DIAGNOSTIC_CEMAC_BEAC.csv : 22 lignes (6 pays x annees).


---
#### Tableau 4 -- detail credit/debit par poste (tous pays x toutes leurs annees)

Complement aux tableaux 1-3 (qui ne montrent que des soldes nets, des ratios, ou la table
brute complete `metriques_cemac_local`) : credit et debit bruts, presentes comme un
tableau comparatif au meme format que les precedents, pour Biens, Services, Revenu
primaire et Revenu secondaire -- les seuls postes pour lesquels ces fichiers BEAC
distinguent credit/debit (IDE et Portefeuille n'y sont disponibles qu'en solde net, cf.
section 4). Meme perimetre pays/annees que les tableaux precedents.


In [12]:
POSTES_DETAIL_CD_LOCAL = [
    ('BIENS', 'Biens'),
    ('SERVICES', 'Services'),
    ('REVENU_PRIMAIRE', 'Revenu primaire'),
    ('REVENU_SECONDAIRE', 'Revenu secondaire'),
]

# Tableau 4 complet (tous pays x toutes leurs annees) -> CSV, credit/debit bruts ---
colonnes_t4_local = ['COUNTRY_CODE', 'PAYS', 'ANNEE']
for prefixe, _ in POSTES_DETAIL_CD_LOCAL:
    colonnes_t4_local += [f'CREDIT_{prefixe}', f'DEBIT_{prefixe}']

lignes_t4_local = []
for code, nom in PAYS_CEMAC_LOCAL.items():
    if code not in ANNEES_PAR_PAYS_LOCAL:
        continue
    for a in ANNEES_PAR_PAYS_LOCAL[code]:
        r = ligne_pour(code, a)
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        for prefixe, _ in POSTES_DETAIL_CD_LOCAL:
            ligne[f'CREDIT_{prefixe}'] = r[f'CREDIT_{prefixe}'] if r is not None else None
            ligne[f'DEBIT_{prefixe}'] = r[f'DEBIT_{prefixe}'] if r is not None else None
        lignes_t4_local.append(ligne)

df_t4_local_complet = pd.DataFrame(lignes_t4_local, columns=colonnes_t4_local)
chemin_t4_local_csv = DIR_SORTIE / 'TABLEAU4_DETAIL_CREDIT_DEBIT_CEMAC_BEAC.csv'
df_t4_local_complet.to_csv(chemin_t4_local_csv, index=False)
print(f"{chemin_t4_local_csv} : {len(df_t4_local_complet)} lignes "
      f"({df_t4_local_complet['COUNTRY_CODE'].nunique()} pays x annees) -- credit/debit bruts par poste, "
      "millions USD (source : releves BEAC, taux fixe XAF/USD). IDE et Portefeuille non repris ici : "
      "seul le solde net est disponible dans ces fichiers BEAC (cf. section 4).")

# Colonnes formatees Credit/Debit par poste ---
COLONNES_T4_LOCAL = []
for prefixe, libelle in POSTES_DETAIL_CD_LOCAL:
    COLONNES_T4_LOCAL.append((f'Credit {libelle}', f'CREDIT_{prefixe}'))
    COLONNES_T4_LOCAL.append((f'Debit {libelle}', f'DEBIT_{prefixe}'))


def tableau_t4_local_formate(df_sub, index_series):
    """Sous-ensemble de df_t4_local_complet -> tableau presentable (colonnes Credit/Debit par
    poste, formatees), avec l'index fourni (annees ou noms de pays) -- pas de transposition :
    meme orientation que le CSV (une ligne par pays x annee)."""
    df_fmt = pd.DataFrame({lbl: df_sub[cle].apply(fmt_local).values for lbl, cle in COLONNES_T4_LOCAL})
    df_fmt.index = index_series.values
    return df_fmt


# Exemple par pays (annees en lignes, postes en colonnes) ---
code_ex4 = next(iter(ANNEES_PAR_PAYS_LOCAL))
nom_ex4 = PAYS_CEMAC_LOCAL[code_ex4]
annees_ex4 = ANNEES_PAR_PAYS_LOCAL[code_ex4]
sub_ex4_local = df_t4_local_complet[df_t4_local_complet['COUNTRY_CODE'] == code_ex4].sort_values('ANNEE')
df_t4_ex4_local = tableau_t4_local_formate(sub_ex4_local, sub_ex4_local['ANNEE'])

fig_t4_ex4_local = render_table_local(
    df_t4_ex4_local,
    title=f'Tableau 4 (exemple)  {nom_ex4}\nDetail credit/debit par poste ({annees_ex4[0]}–{annees_ex4[-1]}, millions USD)',
    caption=f"Exemple illustratif parmi {len(ANNEES_PAR_PAYS_LOCAL)} pays -- serie complete dans "
            f"{chemin_t4_local_csv.name}.",
    fontsize=10,
    filename=FIGURES_DIR / f'tab4_detail_credit_debit_beac_{code_ex4}.png',
)
fig_t4_ex4_local.show()

# Exemple comparatif (pays en lignes, postes en colonnes) ---
annees_comparables_t4_local = [a for a in ANNEES_UNION_LOCAL if len([c for c in ANNEES_PAR_PAYS_LOCAL if a in ANNEES_PAR_PAYS_LOCAL[c]]) >= 2]
if annees_comparables_t4_local:
    annee_ex4 = annees_comparables_t4_local[-1]
    sub_comp4_local = df_t4_local_complet[df_t4_local_complet['ANNEE'] == annee_ex4].sort_values('PAYS')
    df_t4_comp4_local = tableau_t4_local_formate(sub_comp4_local, sub_comp4_local['PAYS'])

    fig_t4_comp4_local = render_table_local(
        df_t4_comp4_local,
        title=f'Tableau 4 comparatif (exemple)  {annee_ex4}\nDetail credit/debit par poste (millions USD)',
        caption=f"Exemple illustratif -- {len(sub_comp4_local)} pays disponibles cette annee-la sur "
                f"{len(ANNEES_PAR_PAYS_LOCAL)} au total ; serie complete dans {chemin_t4_local_csv.name}.",
        fontsize=10,
        filename=FIGURES_DIR / f'tab4_detail_credit_debit_beac_comparatif_{annee_ex4}.png',
    )
    fig_t4_comp4_local.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU4_DETAIL_CREDIT_DEBIT_CEMAC_BEAC.csv : 22 lignes (6 pays x annees) -- credit/debit bruts par poste, millions USD (source : releves BEAC, taux fixe XAF/USD). IDE et Portefeuille non repris ici : seul le solde net est disponible dans ces fichiers BEAC (cf. section 4).
